In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("Economic_data.csv")

In [3]:
import pandas as pd

# Columns to check for missingness
value_cols = [
    'GDP_Growth_Rate',
    'Inflation_Rate',
    'GDP_Per_Capita',
    'Unemployment_Rate',
    'Cost of Living Index',
    'Local Purchasing Power Index'
]

# Create working table
country_year_missing = df[['Country', 'Year'] + value_cols].copy()

# Missing count and index for each country-year row
country_year_missing['Missing_Count'] = country_year_missing[value_cols].isna().sum(axis=1)
country_year_missing['Missingness_Index'] = (
    country_year_missing['Missing_Count'] / len(value_cols) * 100
).round(2)

# Missingness category
def categorize_missing(count):
    if count == 0:
        return 'Complete'
    elif count == 1:
        return 'Low Missingness(1 variables)'
    elif count in [2, 3]:
        return 'Moderate Missingness(2-3 variables)'
    elif count in [4, 5]:
        return 'High Missingness(4-5 Variables)'
    else:
        return 'Fully Missing'


country_year_missing['Missingness_Category'] = country_year_missing['Missing_Count'].apply(categorize_missing)

# Create decade-style year groups like 1970-1979
country_year_missing['Year_Group'] = (
    (country_year_missing['Year'] // 10) * 10
).astype(int).astype(str) + '-' + (
    ((country_year_missing['Year'] // 10) * 10 + 9)
).astype(int).astype(str)

# Overall missingness category summary
category_summary = (
    country_year_missing['Missingness_Category']
    .value_counts()
    .reindex(['Complete', 'Low Missingness(1 variables)', 'Moderate Missingness(2-3 variables)', 'High Missingness(4-5 Variables)', 'Fully Missing'])
    .reset_index()
)
category_summary.columns = ['Missingness_Category', 'Number_of_rows']
category_summary['Percentage (%)'] = (
    category_summary['Number_of_rows'] /
    category_summary['Number_of_rows'].sum() * 100
).round(2)

# Average missingness by decade/year group
missing_by_year_group = (
    country_year_missing.groupby('Year_Group')['Missingness_Index']
    .mean()
    .round(2)
    .reset_index(name='Average_Missingness_Index')
)
missing_by_year_group = missing_by_year_group.sort_values(
    by='Year_Group',
    key=lambda x: x.str[:4].astype(int)
)

# Missingness category summary within each decade/year group
category_by_year_group = (
    country_year_missing.groupby(['Year_Group', 'Missingness_Category'])
    .size()
    .reset_index(name='Count')
)
category_by_year_group['Percentage (%)'] = (
    category_by_year_group.groupby('Year_Group')['Count']
    .transform(lambda x: (x / x.sum() * 100).round(2))
)
category_by_year_group = category_by_year_group.sort_values(
    by='Year_Group',
    key=lambda x: x.str[:4].astype(int)
)

# Print outputs
print("\n=== Overall Missingness Category Summary ===")
print(category_summary.to_string(index=False))

print("\n=== Average Missingness by Year Group ===")
print(missing_by_year_group.to_string(index=False))

print("\n=== Missingness Category by Year Group ===")
print(category_by_year_group.to_string(index=False))


=== Overall Missingness Category Summary ===
               Missingness_Category  Number_of_rows  Percentage (%)
                           Complete          1181.0            9.20
       Low Missingness(1 variables)            41.0            0.32
Moderate Missingness(2-3 variables)          9368.0           72.99
    High Missingness(4-5 Variables)          2245.0           17.49
                      Fully Missing             NaN             NaN

=== Average Missingness by Year Group ===
Year_Group  Average_Missingness_Index
 1970-1979                      57.43
 1980-1989                      55.34
 1990-1999                      39.40
 2000-2009                      36.73
 2010-2019                      28.45
 2020-2029                      30.66

=== Missingness Category by Year Group ===
Year_Group                Missingness_Category  Count  Percentage (%)
 1970-1979     High Missingness(4-5 Variables)    770           41.38
 1970-1979 Moderate Missingness(2-3 variables)   1091